# Agent Memory, Zero to Hero

## Build a memory-first engineering copilot with MemoRizz 0.6

A language model is stateless. An agent becomes continuous only when a
host system deliberately forms, scopes, retrieves, compacts, governs,
and forgets memory. In this notebook we build **Memo**, an engineering
copilot, one memory capability at a time.

The examples use the MemoRizz 0.6 SDK and make each memory boundary
visible through scoped records, assertions, and operational evidence.

**Audience.** This lesson is written for AI engineers, application
developers, forward-deployed engineers, and solution architects who
already know how to call an LLM and now need to make an agent reliable
across turns, users, deployments, and model upgrades.

**What you will build.** By the end, one scoped agent will remember a
conversation, retain a versioned identity, resolve structured facts,
retrieve source material, discover tools progressively, recall a
procedure, load a reviewed skill, reuse safe answers, compact history,
coordinate with another agent, and expose operational evidence.

**What you will learn.** You should be able to explain:

1. why model context is not durable memory;
2. how episodic, semantic, procedural, working, and social memory differ;
3. where tenant scope, provenance, freshness, and forgetting belong;
4. why a memory provider is more than a vector database;
5. how memory changes token use, latency, cost, and answer quality; and
6. how to test a memory system without confusing a plumbing demo with
   a model or retrieval benchmark.

## Design principles used throughout

| Principle | How this notebook applies it |
|---|---|
| Provider portability | Filesystem keeps the lesson self-contained; the memory APIs also support Oracle and MongoDB providers |
| Secret safety | `getpass` requests the OpenAI key only when it is absent from the process environment |
| Explicit scope | Every application turn carries `memory_id`, `user_id`, and `thread_id` |
| Reviewed procedures | `with_skills(...)` and Skillbox retrieve bounded, approved instructions |
| Bounded tool context | `ContextPolicy` and the semantic router disclose only a small relevant tool set |
| Governed cache reuse | Fingerprints, freshness domains, inspection, bypass, and invalidation are explicit |
| Auditable compression | `generate_summaries(...)` receives explicit scope and retains links to source turns |
| Operational evidence | `capability_report()` and `observability_summary()` expose structured runtime state |

OpenAI provides reasoning throughout the notebook. A deterministic local
hash embedder keeps filesystem retrieval repeatable and avoids a second
hosted dependency; it is an instructional retrieval fixture, not a
product-quality embedding benchmark.

The lesson deliberately separates two questions:

- **Did the memory mechanism work?** We answer this with deterministic
  assertions, stored rows, source IDs, counters, and reloads.
- **Did a model answer well?** That requires a representative dataset,
  retrieval metrics, grounded scoring, latency, token, and cost
  accounting. It belongs in an evaluation run, not in a tutorial claim.

## The memory lifecycle

```mermaid
flowchart LR
  E[Experience] --> F[Form memory]
  F --> S[(Scoped provider)]
  S --> R[Retrieve and rank]
  R --> C[Build bounded context]
  C --> A[Reason and act]
  A --> O[Observe outcome]
  O --> K[Consolidate or learn]
  K --> S
  S --> G[Invalidate, compact, or forget]
```

The engineering goal is not to remember everything. It is to preserve
the smallest trustworthy state that improves future behavior while
respecting freshness, ownership, provenance, latency, and cost.

A useful vocabulary:

- **Formation** decides what experience becomes a durable unit.
- **Retrieval** selects evidence before the model sees it.
- **Recall** is successful retrieval; it is not the same as a correct answer.
- **Refinement** updates facts, summaries, skills, and confidence from evidence.
- **Forgetting** removes, expires, demotes, invalidates, or makes a unit
  less retrievable when it is stale, harmful, superseded, or no longer useful.

MemoRizz treats forgetting as policy, not data loss by accident. The
appropriate mechanism depends on the representation: cache entries
expire, skills can be demoted, entity claims can be superseded,
summaries replace repeated prompt inclusion while preserving source
links, and scoped retention can delete data when required.

## The stack we will assemble

| Layer | `MemoryType` | What it contributes |
|---|---|---|
| Episodic | `CONVERSATION_MEMORY`, `SUMMARIES` | What happened, plus source-linked compression |
| Semantic | `PERSONAS`, `ENTITY_MEMORY`, `KNOWLEDGE_BASE` | Identity, canonical facts, and documents |
| Procedural | `TOOLBOX`, `WORKFLOW_MEMORY`, `SKILLBOX`, `TOOL_LOG` | Capabilities, successful procedures, learned instructions, and audit |
| Working | `SHORT_TERM_MEMORY`, `SEMANTIC_CACHE` | Bounded active context and governed response reuse |
| Social | `SHARED_MEMORY` | Multi-agent coordination with ownership and trace scope |
| Durability | `MEMAGENT` | Persisted agent definition and memory links |

These stores are complementary. A sentence from a user can begin as an
episodic turn, become a structured entity attribute after validation,
appear in a source-linked summary, and invalidate a cached answer. That
does not justify copying every statement everywhere: each derived unit
needs a reason, a source, a scope, and a lifecycle owner.

> **Memory versus context:** memory is durable state available for
> future selection. Context is the bounded evidence assembled for one
> model call. Good memory engineering improves context; it does not
> indiscriminately inject the entire database into the prompt.

## 0 · Setup

In a fresh environment, install the local-development extra:

```bash
python -m pip install "memorizz[filesystem]" jupyter
```

When developing from this repository, install it editable instead:

```bash
python -m pip install -e ".[filesystem]"
```

The next cells use OpenAI for every reasoning call. If
`OPENAI_API_KEY` is not already in the process environment, the setup
prompts securely with `getpass` and never stores the value in the notebook.
A small deterministic hash embedder keeps filesystem retrieval repeatable.

**Success signal.** The first output should identify MemoRizz 0.6, the
OpenAI reasoning model, and an isolated temporary filesystem. Rerunning
the notebook does not touch production memory.

### MemoRizz imports used in the lesson

| Import | What it provides | Why it is used here |
|---|---|---|
| `MemAgent`, `MemAgentBuilder` | Agent runtime and fluent construction API | Build, persist, restore, and run the memory-enabled copilot |
| `FileSystemProvider`, `FileSystemConfig` | Local durable memory provider | Keep the lesson isolated while exercising the full provider contract |
| `MemoryType`, `RoleType`, `ApplicationMode` | Stable enums for stores, message roles, and runtime behavior | Avoid fragile string literals at memory boundaries |
| `Persona`, `EntityMemory`, `KnowledgeBase` | Semantic identity, structured facts, and source documents | Demonstrate three distinct forms of semantic memory |
| `Toolbox`, `governed_tool`, `ToolResultPolicy` | Tool registration, policy metadata, and result offloading | Keep capabilities bounded and tool evidence auditable |
| `Workflow`, `SkillStatus` | Durable procedures and reviewed skills | Show how successful actions become reusable procedural memory |
| `ContextPolicy`, `SharedMemory` | Context budgeting and scoped multi-agent coordination | Control retrieval pressure and coordinate without global state |
| `set_global_embedding_manager` | Embedding contract shared by semantic stores | Make retrieval and semantic-cache behavior consistent |
| `OpenAI` | MemoRizz OpenAI reasoning provider | Use one hosted reasoning model throughout the notebook |

In [1]:
import getpass
import hashlib
import math
import os
import tempfile
from importlib.metadata import version
from pathlib import Path
import memorizz
from memorizz import (
    ApplicationMode,
    ContextPolicy,
    EntityMemory,
    FileSystemConfig,
    FileSystemProvider,
    KnowledgeBase,
    MemAgent,
    MemAgentBuilder,
    MemoryType,
    Persona,
    RoleType,
    SharedMemory,
    Toolbox,
    ToolResultPolicy,
    governed_tool,
)
from memorizz.embeddings import set_global_embedding_manager
from memorizz.long_term.procedural.skillbox import SkillStatus
from memorizz.long_term.procedural.workflow import Workflow
from memorizz.llms.openai import OpenAI


def ensure_secret(name: str, prompt: str) -> None:
    if os.getenv(name):
        return
    value = getpass.getpass(prompt).strip()
    if not value:
        raise RuntimeError(f"{name} is required to run this notebook.")
    os.environ[name] = value


ensure_secret("OPENAI_API_KEY", "Enter your OpenAI API key: ")
OPENAI_MODEL = os.getenv("MEMORIZZ_OPENAI_MODEL", "gpt-5.6-luna")

RUN_ID = os.getenv("MEMORIZZ_RUN_ID", "zero-to-hero")
MEMORY_ID = f"engineering-copilot-{RUN_ID}"
USER_ID = "ada"
THREAD_ID = "rag-migration"
NOTEBOOK_ROOT = Path(tempfile.mkdtemp(prefix="memorizz-agent-memory-"))

print(
    {
        "memorizz_version": version("memorizz"),
        "package": "memorizz",
        "reasoning_provider": "OpenAI",
        "reasoning_model": OPENAI_MODEL,
        "memory_backend": "isolated temporary filesystem",
    }
)

{'memorizz_version': '0.6.0', 'package': 'memorizz', 'reasoning_provider': 'OpenAI', 'reasoning_model': 'gpt-5.6-luna', 'memory_backend': 'isolated temporary filesystem'}


### A deterministic embedding for repeatable retrieval

Production semantic retrieval should use a validated embedding model.
For this notebook, token hashing gives us repeatable vectors with no
download or network dependency. The class implements the same small
contract used by the filesystem provider and semantic cache.

This hash embedder is intentionally weak: shared words can produce a
useful demonstration ranking, but it does not model paraphrase or intent.
Never interpret its retrieval scores as product quality. Its value here
is experimental control—if an assertion changes, the memory code or
fixture changed, not a downloaded model snapshot.

The provider capability output tells us which guarantees this backend
offers. `native_vector_search=False` means the filesystem provider will
use its portable exact-search path in this configuration. The same
application APIs can later target Oracle or MongoDB, but capability and
performance differences still need to be measured.

In [2]:
class LocalHashEmbeddings:
    dimensions = 64

    def get_embedding(self, text: str, **_kwargs):
        vector = [0.0] * self.dimensions
        for token in str(text).lower().split():
            digest = hashlib.sha256(token.encode("utf-8")).digest()
            vector[int.from_bytes(digest[:2], "big") % self.dimensions] += 1.0
        norm = math.sqrt(sum(value * value for value in vector)) or 1.0
        return [value / norm for value in vector]

    def get_embeddings(self, texts, **kwargs):
        return [self.get_embedding(text, **kwargs) for text in texts]

    def get_dimensions(self):
        return self.dimensions

    def get_default_model(self):
        return "local-hash-v1"

    def get_provider_info(self):
        return {
            "provider": "local-hash",
            "model": self.get_default_model(),
            "dimensions": self.dimensions,
            "config": {},
        }

In [3]:

embeddings = LocalHashEmbeddings()
set_global_embedding_manager(embeddings)

provider = FileSystemProvider(
    FileSystemConfig(
        root_path=NOTEBOOK_ROOT,
        use_faiss=False,
        embedding_provider=embeddings,
    )
)
print(provider.memory_capabilities().to_dict())

{'provider': 'FileSystemProvider', 'batch_store': True, 'transactional_batch': False, 'scoped_search': True, 'result_scores': True, 'provenance': True, 'native_vector_search': False, 'native_hybrid_search': False}


### Configure the OpenAI reasoning model

Every reasoning call uses MemoRizz's `OpenAI` provider. The factory keeps
model construction consistent for the raw baseline, persisted agents,
tool use, semantic cache, summarization, and restored agents. Generated
wording may vary, so the notebook verifies stored records, scope, cache
statistics, and retrieval evidence instead of matching exact prose.

In [4]:
def make_model():
    return OpenAI(
        api_key=os.environ["OPENAI_API_KEY"],
        model=OPENAI_MODEL,
        reasoning_effort="none",
    )


print(make_model().get_config())

{'provider': 'openai', 'model': 'gpt-5.6-luna', 'reasoning_effort': 'none'}


---
# 1 · Feel the statelessness

A model call remembers only the messages supplied in that call. Two
independent calls do not form a conversation.

**Experiment.** The first request states Ada's identity and project;
the second request contains neither. A raw provider receives two
unrelated message arrays, so the correct behavior is to admit that the
information is unavailable.

**Read the output.** The first response acknowledges the statement. The
second call receives no previous messages, so its answer cannot rely on
Ada's identity or project. This establishes the stateless baseline before
the provider begins replaying scoped evidence.

> **Checkpoint:** an LLM API, a chat UI, and an agent are different
> layers. A chat UI may replay earlier messages; the underlying model
> still sees only the context sent for the current inference.

In [5]:
raw_model = make_model()
first = raw_model.generate(
    [{"role": "user", "content": "I'm Ada and I am migrating our RAG stack to Oracle."}]
)
second = raw_model.generate(
    [{"role": "user", "content": "What project am I working on and who am I?"}]
)
print("First call :", first)
print("Second call:", second)

assert first and second

First call : Hi Ada! Oracle can support a RAG stack well, especially if you want vector search, relational metadata, security, and transactional workflows in one platform.

A typical Oracle-based RAG architecture includes:

1. **Document ingestion**
   - Extract text and metadata from PDFs, HTML, Office files, and databases.
   - Chunk documents with overlap and preserve source, permissions, timestamps, and version information.

2. **Embeddings**
   - Generate embeddings using an Oracle-supported embedding model or an external provider.
   - Store vectors in Oracle’s vector data type and index them for similarity search.

3. **Hybrid retrieval**
   - Combine semantic vector search with keyword, metadata, and structured SQL filters.
   - Apply tenant and authorization predicates during retrieval rather than after it.

4. **Reranking and context assembly**
   - Rerank the top candidates, deduplicate overlapping chunks, and fit the final context to the model’s token budget.

5. **Generati

---
# 2 · Episodic memory: conversation and durable agent state

`MemAgent.run()` owns the conversational write path. It combines exact
host-owned scope with retrieval, model execution, trace capture, and
persistence. `build_and_save()` also writes the agent definition to
`MEMAGENT` so another process can restore it.

**Key term — episodic memory.** Episodic memory records events in time:
who said what, in which thread, for which tenant and agent. Its unit is
a role/content pair plus timestamp, identifiers, provenance, optional
embedding, and later an optional summary marker.

```mermaid
sequenceDiagram
  participant H as Trusted host
  participant A as MemAgent
  participant P as Memory provider
  participant M as Model
  H->>A: run(query, memory_id, user_id, thread_id)
  A->>P: retrieve scoped history and memory
  P-->>A: bounded evidence
  A->>M: instruction + context + current query
  M-->>A: response
  A->>P: persist user/assistant turns and trace metadata
  A-->>H: response
```

**Read the output.** The second agent turn can identify Ada, the history
contains both user and assistant rows, and a newly reconstructed agent
recalls the project. Together those checks prove turn persistence and
process-level reconstruction; response text alone would not.

In [6]:
copilot = (
    MemAgentBuilder()
    .with_name(f"Memo-{RUN_ID}")
    .with_instruction(
        "You are Memo, a concise engineering copilot. Ground claims in retrieved memory."
    )
    .with_model(make_model())
    .with_memory_provider(provider)
    .with_memory_ids(MEMORY_ID)
    .with_application_mode(ApplicationMode.ASSISTANT.value)
    .with_automations_enabled(False)
    .build_and_save()
)

scope = {"memory_id": MEMORY_ID, "user_id": USER_ID, "thread_id": THREAD_ID}
print(copilot.run("I'm Ada. I am migrating our RAG stack to Oracle AI Database.", **scope))
print(copilot.run("What project am I working on and who am I?", **scope))

Tool execution failed: 1 validation error for EntityAttribute
name
  Field required [type=missing, input_value={'attribute': 'name', 'value': 'Ada'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


Nice to meet you, Ada. I’ll keep in mind that you’re migrating your RAG stack to Oracle AI Database.


You’re Ada, and you’re migrating your RAG stack to Oracle AI Database.


In [7]:
history = provider.retrieve_conversation_history_ordered_by_timestamp(
    memory_id=MEMORY_ID,
    memory_type=MemoryType.CONVERSATION_MEMORY,
    user_id=USER_ID,
    thread_id=THREAD_ID,
)
print([(row.get("role"), row.get("content")) for row in history])
assert {row.get("role") for row in history} == {"user", "assistant"}

restored = MemAgent.load(
    copilot.agent_id,
    memory_provider=provider,
    model=make_model(),
)
print(restored.run("What project were we discussing?", **scope))

[('user', "I'm Ada. I am migrating our RAG stack to Oracle AI Database."), ('assistant', 'Nice to meet you, Ada. I’ll keep in mind that you’re migrating your RAG stack to Oracle AI Database.'), ('user', 'What project am I working on and who am I?'), ('assistant', 'You’re Ada, and you’re migrating your RAG stack to Oracle AI Database.')]


We were discussing your migration of the RAG stack to Oracle AI Database.


**Why all three scope keys matter**

- `memory_id` identifies the application or durable memory space.
- `user_id` is the tenant boundary. It must come from trusted host state.
- `thread_id` separates conversations inside that memory space.

Omitting a tenant filter is not equivalent to `user_id=None`: `None`
means the anonymous/legacy tenant, while omission is an administrative,
potentially unscoped read on provider APIs that support it.

Scope must come from authenticated host state—not from a model-produced
tool argument. In a multi-tenant service, attach these identifiers at
the request boundary and test that a query for user A cannot retrieve
user B. Thread scope prevents unrelated conversations for the same user
from contaminating one another.

**Production decision:** conversation memory is appropriate when the
chronology matters. Promote stable facts into entity memory and compress
old turns into summaries rather than treating an indefinitely growing
transcript as the only source of truth.

---
# 3 · Semantic memory: identity and canonical facts

Conversation memory records what was said. Semantic memory represents
what should be treated as durable knowledge: an agent identity, an
entity profile, or an application document.

| Representation | Question answered | Update semantics |
|---|---|---|
| Persona | Who is this agent and what is its role? | Versioned evolution with a reason |
| Entity | What is currently known about this person, service, or object? | Attribute-level source and confidence |
| Knowledge base | What do approved documents say? | Re-ingest/version source material |

The code first evolves Memo's goal and records the trigger. It then
creates a structured Ada entity with independently sourced attributes.
The printed persona version and entity attributes are the evidence to
inspect. Neither representation should be silently rewritten from an
untrusted model guess.

**Forgetting and correction.** A persona change should remain
auditable. An entity fact should be superseded or removed when its
source changes. Keeping provenance allows retrieval policy to prefer a
current service catalog over an older conversational mention.

In [8]:
persona = Persona(
    name="Memo",
    role=RoleType.TECHNICAL_EXPERT,
    goals="Help engineers ship reliable, memory-first AI systems.",
    background="A staff AI platform engineer focused on retrieval and observability.",
)
copilot.set_persona(persona)

evolution = persona.update(
    updates={"goals": "Help engineers ship reliable, grounded, and cost-aware AI systems."},
    change_trigger={
        "reason": "The platform team adopted explicit cost budgets.",
        "source_type": "user_feedback",
        "source_id": "tutorial-governance-decision",
        "agent_id": copilot.agent_id,
    },
    provider=provider,
)
print({"persona_version": persona.version, "updated": evolution["updated"]})

{'persona_version': 2, 'updated': True}


In [9]:
entities = EntityMemory(provider)
entities.upsert_entity(
    entity_id="user-ada",
    name="Ada",
    entity_type="person",
    attributes=[
        {"name": "project", "value": "Oracle AI Database RAG migration", "confidence": 0.98, "source": "user"},
        {"name": "communication_style", "value": "concise technical guidance", "confidence": 0.95, "source": "user"},
    ],
    memory_id=MEMORY_ID,
    user_id=USER_ID,
)
entities.record_attribute(
    entity_id="user-ada",
    attribute_name="owns_service",
    attribute_value="retrieval-api",
    confidence=0.9,
    source="team-directory",
    memory_id=MEMORY_ID,
    user_id=USER_ID,
)
ada = entities.get_entity("user-ada", memory_id=MEMORY_ID, user_id=USER_ID)
print([(item["name"], item["value"]) for item in ada["attributes"]])

[('project', 'Oracle AI Database RAG migration'), ('communication_style', 'concise technical guidance'), ('owns_service', 'retrieval-api')]


---
# 4 · Semantic memory: knowledge-base grounding

A knowledge base stores source-linked documents and chunks. Ingestion
and retrieval are separate from answer generation, which lets us test
retrieval before asking an LLM to synthesize anything.

```mermaid
flowchart LR
  D[Source document] --> C[Chunk]
  C --> E[Embed and index]
  Q[Question] --> R[Scoped retrieval]
  E --> R
  R --> G[Grounded generation]
  G --> V[Citation and entailment check]
```

The assertion checks that the rollback passage reaches the evidence
set. That is a **retrieval test**, not an answer-quality test. In
production, measure recall/MRR separately from grounded answer quality;
a fluent answer can be wrong when retrieval misses, and a weak reader
can still mishandle correctly retrieved evidence.

**When to use it:** policies, runbooks, contracts, manuals, and other
source documents. **When not to use it:** a single mutable field such
as the current owner of a service—that belongs in entity/state memory.

In [10]:
runbook = (
    "Premium support includes unlimited vector storage.\n\n"
    "Before rebuilding an Oracle vector index, take a schema snapshot, verify free "
    "space, announce the maintenance window, and preserve a tested rollback path."
)

kb = KnowledgeBase(memory_provider=provider)
kb_id = kb.ingest_knowledge(
    corpus=runbook,
    namespace="platform-runbook",
    chunking_strategy="paragraph",
    user_id=USER_ID,
)
kb.attach_to_agent(copilot, kb_id)

hits = provider.retrieve_by_query(
    "tested rollback path for the vector index",
    memory_store_type=MemoryType.KNOWLEDGE_BASE,
    namespace="platform-runbook",
    user_id=USER_ID,
    limit=3,
)
print([(round(hit.get("score", 0.0), 3), hit["content"]) for hit in hits])
assert any("rollback" in hit["content"].lower() for hit in hits)

[(0.526, 'Before rebuilding an Oracle vector index, take a schema snapshot, verify free space, announce the maintenance window, and preserve a tested rollback path.'), (0.154, 'Premium support includes unlimited vector storage.')]


---
# 5 · Procedural memory: tools, workflows, and skills

Procedural memory answers **how should the agent act?**

- Toolbox stores strict capability schemas and trusted callable bindings.
- Workflow memory records what was actually executed.
- Skillbox stores reviewed instructions distilled or authored for reuse.
- Tool logs preserve auditable outputs and offload large results by policy.

| Memory | Representation | Authority | Typical failure |
|---|---|---|---|
| Toolbox | Callable plus strict schema and policy | Trusted host registry | Exposing too many or unsafe tools |
| Workflow | Ordered observed or authored steps | Runtime/application | Replaying stale arguments blindly |
| Skillbox | Retrieved instruction/playbook | Reviewed author or promotion policy | Injecting an irrelevant instruction |
| Tool log | Immutable execution evidence | Tool runtime | Logging secrets or recursively offloading pointers |

`ContextPolicy(progressive_tool_disclosure=True)` keeps the full catalog
out of every model request. The router retrieves a small allowlist for
this turn, then performs strict binding before dispatch. This reduces
prompt tokens and accidental tool choice without allowing the model to
invoke an undisclosed name.

The health tool returns a deliberately large diagnostic payload.
`ToolResultPolicy` keeps small outputs inline but stores this full result
exactly once and sends an auditable pointer/digest back to the model.
The printed router preview and `TOOL_LOG` row demonstrate both stages.

In [11]:
@governed_tool(deterministic=True, side_effects=False, domains=("platform-status",))
def system_status() -> dict:
    '''Return health plus a bounded diagnostic report for the platform.'''
    return {
        "platform": "healthy",
        "vector_index": "ready",
        "diagnostics": [
            f"check-{index:02d}: healthy" for index in range(24)
        ],
    }


toolbox = Toolbox.from_functions(
    [system_status],
    memory_provider=provider,
    user_id=USER_ID,
    augment=False,
)

ops_agent = (
    MemAgentBuilder()
    .with_name(f"Memo-Ops-{RUN_ID}")
    .with_instruction("Use read-only tools when they provide fresher evidence.")
    .with_model(make_model())
    .with_memory_provider(provider)
    .with_memory_ids(MEMORY_ID)
    .with_tools([system_status])
    .with_toolbox(toolbox)
    .with_context_policy(ContextPolicy(progressive_tool_disclosure=True, tool_top_k=2))
    .with_tool_result_policy(ToolResultPolicy(offload_above_chars=256))
    .with_automations_enabled(False)
    .build_and_save()
)

print("Router preview:", ops_agent.semantic_tool_router.preview("Check system status", user_id=USER_ID))
print(ops_agent.run("Check the system status.", **scope))
tool_logs = ops_agent.memory_manager.list_tool_logs(
    MEMORY_ID, user_id=USER_ID, thread_id=THREAD_ID
)
print("Tool logs:", [(row.get("tool_name"), row.get("success")) for row in tool_logs])
assert any(row.get("tool_name") == "system_status" for row in tool_logs)

Router preview: ['system_status']


System status: **healthy**

- Platform: healthy
- Vector index: ready
- Diagnostics: all reported checks healthy
Tool logs: [('system_status', True)]


In [12]:
workflow = Workflow(
    name="safe-vector-index-rebuild",
    description="Rebuild an index with explicit verification and rollback.",
    memory_id=MEMORY_ID,
    agent_id=copilot.agent_id,
    user_id=USER_ID,
    user_query="How should I rebuild the vector index?",
)
workflow.add_step("inspect", {"tool": "system_status", "arguments": {}})
workflow.add_step("snapshot", {"tool": "create_schema_snapshot", "arguments": {"scope": "vector"}})
workflow.add_step("verify", {"tool": "verify_index", "arguments": {"mode": "exact"}})
workflow_row_id = workflow.store_workflow(provider)

recalled_workflows = Workflow.retrieve_workflows_by_query(
    "safe vector index rebuild", provider, limit=2
)
print(
    {
        "stored_row": workflow_row_id,
        "canonical_hash": recalled_workflows[0].canonical_hash,
        "step_count": recalled_workflows[0].step_count,
    }
)

{'stored_row': 'bbcb08c5-4472-4e3d-b8bd-8b268565b1f7', 'canonical_hash': '694599fe46d56196beba37b9669f8f630780cccdd8b6f82644ebf94b903dc688', 'step_count': 3}


In [13]:
authored_skill = {
    "name": "platform/vector-index-rebuild",
    "description": "Safely rebuild a production vector index.",
    "content": (
        "Check health, snapshot schema state, verify capacity, announce the window, "
        "rebuild, validate exact search, then retain a tested rollback path."
    ),
    "preconditions": ["production access is approved", "rollback is tested"],
    "tools_used": ["system_status"],
    "queries": ["rebuild vector index", "repair vector search index"],
    "user_id": USER_ID,
    "status": SkillStatus.ACTIVE.value,
}

skilled_agent = (
    MemAgentBuilder()
    .with_name(f"Memo-Skilled-{RUN_ID}")
    .with_instruction("Retrieve a relevant authored skill before proposing a procedure.")
    .with_model(make_model())
    .with_memory_provider(provider)
    .with_memory_ids(MEMORY_ID)
    .with_skills([authored_skill], persistence="skillbox")
    .with_skill_retrieval(enabled=True, top_k=2, min_similarity=0.0)
    .with_continual_learning(enabled=False)
    .with_automations_enabled(False)
    .build_and_save()
)

skill_hits = skilled_agent.skillbox.retrieve_skills_by_query(
    "How do I rebuild vector search safely?",
    limit=2,
    min_similarity=0.0,
    user_id=USER_ID,
)
print([(hit.skill.name, round(hit.similarity, 3)) for hit in skill_hits])

[('platform/vector-index-rebuild', 0.507)]


Authored skill retrieval is independent of continual learning. Enabling
continual learning adds an evidence-controlled lifecycle:

```mermaid
flowchart LR
  W[Successful workflow traces] --> C[Canonical trajectory class]
  C --> P[Candidate skill]
  P --> S[Shadow evaluation]
  S -->|approved| A[Active skill]
  A --> M[Outcome monitoring]
  M -->|drift| D[Demoted or revised]
```

Observed model behavior never silently becomes instruction authority.
Promotion is evidence-based, versioned, reviewable, and reversible.

This matters because of instruction hierarchy: a retrieved skill is not
merely evidence; it can influence what the model does. MemoRizz
therefore separates authored skill retrieval from continual learning.
Repeated workflows may propose a candidate, but promotion should depend
on verified outcomes, shadow comparison, minimum support, and host
policy. Drift or negative outcomes can demote the skill without erasing
the underlying workflow evidence.

> **Checkpoint:** a tool does something, a workflow records a sequence,
> and a skill tells the agent how to approach a class of situations.
> Use all three only when those distinct representations are useful.

---
# 6 · Working memory: semantic cache and context control

Semantic similarity is not freshness. MemoRizz cache identity includes
model, prompt, tool schema, completion policy, data version, request
context, user, and session. Side-effecting or non-deterministic tool
candidates bypass cache admission by default.

A semantic cache is an optimization layer, not general long-term
memory. It reduces hosted generation latency and cost only when reuse is
safe. A correct admission rule normally requires a deterministic,
read-only request; a matching tenant/session; compatible fingerprints;
an unexpired TTL; and a current domain/data version.

**Read the output.** The repeated query should produce one model call,
one miss/write, and one hit. Inspection reveals the matched query,
similarity, age, TTL, hit count, and invalidation domains without
exposing cached response content. Domain invalidation then removes the
entry, proving that changed policy data can force regeneration.

**Do not cache:** mutations, browser actions, approval decisions,
volatile inventory, or answers whose freshness cannot be represented.

In [14]:
cache_model = make_model()
cache_agent = (
    MemAgentBuilder()
    .with_name(f"Memo-Cache-{RUN_ID}")
    .with_instruction("Answer deterministic read-only policy questions concisely.")
    .with_model(cache_model)
    .with_memory_provider(provider)
    .with_memory_ids(f"cache-{RUN_ID}")
    .with_semantic_cache(enabled=True, threshold=0.95, scope="session")
    .with_automations_enabled(False)
    .build()
)
cache_scope = {
    "memory_id": f"cache-{RUN_ID}",
    "user_id": USER_ID,
    "thread_id": "policy",
    "context": {"cache_domains": ["policy"], "data_version": "2026-08-23"},
}
query = "What is the capital of France?"
print(cache_agent.run(query, **cache_scope))
print(cache_agent.run(query, **cache_scope))
cache_stats = cache_agent.semantic_cache_stats()
print(cache_stats)

inspection = cache_agent.inspect_semantic_cache(
    query,
    user_id=USER_ID,
    thread_id="policy",
    context=cache_scope["context"],
)
print(inspection.to_dict())
assert cache_stats["hits"] >= 1
assert inspection.hit is True

Paris.
Paris.
{'enabled': True, 'hits': 1, 'misses': 1, 'bypasses': 0, 'writes': 1, 'evictions': 0, 'size': 1, 'bypass_reasons': {}, 'last_hit': {'cache_key': '1042a8a4-ffb1-5fce-8a64-0de4f14fbfeb', 'query': 'What is the capital of France?', 'similarity': 1.0, 'age_seconds': 0.0061719417572021484, 'agent_id': '7aa55e8d-7b6b-5830-87bc-efbd940a1bfa', 'memory_id': 'cache-zero-to-hero', 'session_id': 'policy', 'user_id': 'ada', 'metadata': {'fingerprints': {'model': '1435914a1b0db330d19975d7185255107546f7f6fe4ccd7b8a8f90e7203f6314', 'prompt': '7ad9098d3eb8dfea43074f5f235cdd14c10f0d411554eadac0f2906a4258ad4c', 'tool_schema': 'c8177d5db0a1484f7e2868070e486b7d1c0c4f0ab5d86f7263c8bb4540bc13eb', 'completion_policy': 'e0b81db2190bb1955538db6433846a0880923ce028738c1951fd21fc4edad1b0', 'data_version': '2026-08-23', 'request_context': 'f8aaaa1e69f0828e25a582c1f31dfa361a98ebbf068d151fdaf41ac015729339'}, 'domain': 'policy', 'domains': ['policy'], 'tags': [], 'admission': {'deterministic': True, 'read

In [15]:
removed = cache_agent.invalidate_semantic_cache(domains=["policy"])
print({"invalidated": removed, "stats": cache_agent.semantic_cache_stats()})

{'invalidated': 1, 'stats': {'enabled': True, 'hits': 1, 'misses': 1, 'bypasses': 0, 'writes': 1, 'evictions': 1, 'size': 0, 'bypass_reasons': {}, 'last_hit': {'cache_key': '1042a8a4-ffb1-5fce-8a64-0de4f14fbfeb', 'query': 'What is the capital of France?', 'similarity': 1.0, 'age_seconds': 0.0061719417572021484, 'agent_id': '7aa55e8d-7b6b-5830-87bc-efbd940a1bfa', 'memory_id': 'cache-zero-to-hero', 'session_id': 'policy', 'user_id': 'ada', 'metadata': {'fingerprints': {'model': '1435914a1b0db330d19975d7185255107546f7f6fe4ccd7b8a8f90e7203f6314', 'prompt': '7ad9098d3eb8dfea43074f5f235cdd14c10f0d411554eadac0f2906a4258ad4c', 'tool_schema': 'c8177d5db0a1484f7e2868070e486b7d1c0c4f0ab5d86f7263c8bb4540bc13eb', 'completion_policy': 'e0b81db2190bb1955538db6433846a0880923ce028738c1951fd21fc4edad1b0', 'data_version': '2026-08-23', 'request_context': 'f8aaaa1e69f0828e25a582c1f31dfa361a98ebbf068d151fdaf41ac015729339'}, 'domain': 'policy', 'domains': ['policy'], 'tags': [], 'admission': {'deterministic

---
# 7 · Episodic compression: summaries with lossless links

Summaries reduce prompt size; they do not erase the original messages.
The summary stores `source_message_ids`, time boundaries, and a unit
count, while original messages receive a `summary_id` marker.

Summarization addresses **tokenomics**: old detail need not be resent on
every turn, so prompt tokens, transfer time, and inference latency can
fall. Compression is useful only while important facts and provenance
survive. MemoRizz keeps lossless links so an application can expand a
summary, audit its sources, or regenerate it under a better policy.

The output should show one summary ID, the summary text, source IDs,
time boundaries, and equal source/unit counts. The equality assertion
checks structural integrity—not whether the prose is a perfect summary.

**Summarization versus forgetting:** summarization changes the default
representation presented to the model; retention deletion removes data.
Keep these policies separate for audit, privacy, and debugging.

In [16]:
summary_ids = copilot.generate_summaries(
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id=THREAD_ID,
    days_back=7,
    max_memories_per_summary=20,
)
print("Summary IDs:", summary_ids)

if summary_ids:
    summary = copilot.fetch_context_summary(
        summary_ids[0],
        memory_id=MEMORY_ID,
        user_id=USER_ID,
        thread_id=THREAD_ID,
    )
    print(
        {
            "summary": summary["content"],
            "source_message_ids": summary["source_message_ids"],
            "memory_units_count": summary["memory_units_count"],
            "period": [summary["period_start"], summary["period_end"]],
        }
    )
    assert summary["memory_units_count"] == len(summary["source_message_ids"])

Summary IDs: ['80ba3ed5-5a6f-4375-a85b-a52df4dcb306']
{'summary': '- **Identity and project:** The user is Ada, working on migrating a Retrieval-Augmented Generation (RAG) stack to Oracle AI Database. This project was consistently recalled across multiple exchanges.\n- **Emotionally significant moments/interactions:** The interactions were straightforward and task-focused, with a friendly introduction and successful confirmation of Ada’s identity and project. No strong emotional events or interpersonal tensions were expressed.\n- **Context and patterns:** Conversations centered on project continuity and verification—Ada repeatedly asked what project was being discussed, and the assistant consistently identified the Oracle AI Database RAG migration. A system-health check was also performed in the context of the migration.\n- **Achievements and learning:** The migration context was clearly established and retained. The system-status check confirmed that the relevant platform infrastructu

---
# 8 · Shared memory: a scoped coordination blackboard

Shared memory is not a global scratchpad. A session is owned by a
workflow and tenant, and every command, status, and report identifies
its contributing agent.

The blackboard uses typed messages rather than an unstructured blob:
the lead issues a command, a delegate reports status, and the delegate
returns findings with citations. This makes partial failure,
dependencies, ownership, and provenance inspectable.

Use shared memory when independent agents must coordinate around one
workflow. Do not use it merely to let every agent see every tenant's
state. Scope the session, bound its lifetime, and require citations when
reports rely on retrieved evidence.

In [17]:
shared = SharedMemory(provider)
shared_id = shared.create_shared_session(
    root_agent_id="lead",
    delegate_agent_ids=["researcher", "reviewer"],
    workflow_id=f"vector-review-{RUN_ID}",
    user_id=USER_ID,
    trace_id=f"trace-{RUN_ID}",
)
shared.post_command(
    shared_id,
    agent_id="lead",
    command_id="research-1",
    target_agent_id="researcher",
    instructions="Compare index rebuild options and cite evidence.",
)
shared.post_status(
    shared_id,
    agent_id="researcher",
    command_id="research-1",
    status="completed",
    progress=100,
)
shared.post_report(
    shared_id,
    agent_id="researcher",
    command_id="research-1",
    findings="Use the bounded rebuild procedure with a tested rollback.",
    citations=[summary_ids[0]] if summary_ids else [],
)
print(shared.get_blackboard_entries(shared_id))

[{'memory_id': '6ef169a7-221d-4b13-92de-153adc93a341', 'agent_id': 'lead', 'content': {'message_id': '8eae0a7d-4272-4531-8d55-03738000ba60', 'message_type': 'COMMAND', 'created_at': '2026-08-24T15:40:21.734675', 'payload': {'command_id': 'research-1', 'target_agent_id': 'researcher', 'instructions': 'Compare index rebuild options and cite evidence.', 'priority': 3, 'dependencies': [], 'metadata': {}}}, 'entry_type': 'COMMAND', 'created_at': '2026-08-24T16:40:21.786733'}, {'memory_id': 'f3e461da-c81d-448a-863f-13951563da30', 'agent_id': 'researcher', 'content': {'message_id': '9805d9c1-14d8-45fd-93ff-263e0d401e5a', 'message_type': 'STATUS', 'created_at': '2026-08-24T15:40:21.787441', 'payload': {'command_id': 'research-1', 'agent_id': 'researcher', 'status': 'completed', 'progress': 100, 'blockers': None, 'summary_ids': []}}, 'entry_type': 'STATUS', 'created_at': '2026-08-24T16:40:21.787511'}, {'memory_id': '4d64cf3a-7425-4c69-a82e-18c16eb0bb5c', 'agent_id': 'researcher', 'content': {'m

---
# 9 · Operate the memory system

A production memory layer needs evidence, not just retrieval demos.
Capability and observability reports expose provider features, memory
counts, tool outcomes, summaries, cache behavior, and context usage
without requiring application code to scan raw rows.

Treat this report as a deployment assertion. It answers which provider
and features are active, whether conversations were summarized, what
the latest context cost looked like, and whether tool/workflow failures
occurred. It intentionally reports counts and identifiers rather than
dumping tenant content.

Production dashboards should add p50/p95 retrieval and generation
latency, prompt/completion tokens, cache hit and bypass rates, retrieval
recall, grounded-answer rate, stale-memory incidents, and storage/retention
pressure. A successful SDK call is not an operational SLO.

In [18]:
capabilities = copilot.capability_report()
operations = copilot.observability_summary(
    MEMORY_ID,
    USER_ID,
    thread_id=THREAD_ID,
)
print("Provider:", capabilities["agent"]["memory_provider"])
print("Active memory types:", [item.value for item in copilot.active_memory_types])
print("Conversation:", operations["conversation"])
print("Summaries:", operations["summaries"])
print("Context window:", operations["context_window"])

Provider: FileSystemProvider
Active memory types: ['conversation_memory', 'knowledge_base', 'personas', 'entity_memory', 'short_term_memory', 'summaries']
Conversation: {'row_count': 6, 'message_count': 6, 'role_counts': {'user': 3, 'assistant': 3}, 'thread_count': 1, 'summarized_count': 6, 'trace_bundle_count': 0, 'trace_event_count': 0, 'first_timestamp': 1787582421.719443, 'last_timestamp': 1787582421.723326}
Summaries: {'count': 1}
Context window: {'timestamp': '2026-08-24T16:40:21.717971', 'prompt_tokens': 530, 'completion_tokens': 227, 'total_tokens': 757, 'context_window_tokens': 1050000, 'percentage_used': 0.07209523809523809, 'stage': 'memory_compression'}


## Provider portability

Filesystem is appropriate for tutorials, local agents, and lightweight
deployments. The application-facing APIs remain the same when the
provider changes.

| Provider | Good fit | Operational question to answer first |
|---|---|---|
| Filesystem | Local development, single-node agents, CI fixtures | How will files be backed up and shared safely? |
| MongoDB | Existing document-oriented platforms and horizontal application integration | Are indexes, tenant filters, and vector capabilities configured consistently? |
| Oracle AI Database | Transactional enterprise memory with relational metadata and vector search | Do schema dimensions, privileges, PDB state, and vector-memory policy pass preflight? |

```python
# Oracle AI Database, including runtime readiness and preflight
agent = (
    MemAgentBuilder()
    .with_name("Memo-Oracle")
    .with_instruction("...")
    .with_llm_config({"provider": "openai", "model": "gpt-5.6-luna"})
    .with_oracle_from_env(index_policy="lazy")
    .with_memory_ids("engineering-copilot")
    .build_and_save()
)

# MongoDB
from memorizz import MongoDBConfig, MongoDBProvider
provider = MongoDBProvider(MongoDBConfig.from_env())
```

Validate provider capabilities and retrieval quality before production
rollout; persistence parity does not guarantee identical index behavior.

Provider portability means the memory contracts and scope travel. It
does not mean latency, indexing, transactionality, or ranking are
identical. Run the same provider contract tests and representative
retrieval evaluation for every supported backend.

## Recap

| Need | Use | Primary operation |
|---|---|---|
| Conversation continuity | Episodic memory | `agent.run(...)` |
| Affordable long histories | Summaries | `generate_summaries(...)` |
| Stable agent identity | Persona | `set_persona(...)`, `Persona.update(...)` |
| Canonical user/system facts | Entity memory | `upsert_entity(...)` |
| Grounding in documents | Knowledge base | `ingest_knowledge(...)`, scoped retrieval |
| Discoverable capabilities | Toolbox | `Toolbox.from_functions(...)` |
| Replayable procedures | Workflow memory | `Workflow.store_workflow(...)` |
| Governed reusable instructions | Skillbox | `with_skills(...)`, skill lifecycle |
| Cost and latency reduction | Semantic cache | inspect, invalidate, monitor |
| Multi-agent coordination | Shared memory | command/status/report messages |
| Operational evidence | Reports and traces | `observability_summary(...)` |

Memory engineering is the discipline of choosing, shaping, scoping,
retrieving, governing, measuring, and forgetting these representations.

### Production-readiness checklist

- Derive `user_id` and authorization scope from the trusted host.
- Record source and confidence for durable facts.
- Separate retrieval metrics from answer-generation metrics.
- Admit only safe, fresh responses to semantic cache.
- Require durable approval for side effects.
- Promote workflows into skills only from verified outcomes.
- Preserve summary-to-source expansion.
- Define TTL, supersession, demotion, and deletion policies.
- Monitor tokens, latency, cost, cache behavior, retrieval, and grounding.
- Test reload and cleanup—not just an in-process happy path.

### Suggested exercises

1. Change `THREAD_ID` and prove that Ada's first thread does not leak.
2. Add a second entity source with lower confidence and define a
   deterministic resolution rule.
3. Change the cache `data_version` and explain why it misses.
4. Replace the hash embedder with your production embedder and measure
   retrieval recall on ten questions before changing the reader model.
5. Run the Oracle-backed variant and compare capability/preflight output.

In [19]:
for agent in (restored, copilot, ops_agent, skilled_agent, cache_agent):
    agent.close(close_memory_provider=False)
provider.close()
print("Notebook resources closed; isolated temporary files may now be removed.")

Notebook resources closed; isolated temporary files may now be removed.
